# CourtListener → DataFrame (HCDE 530)

Reads **`COURTLISTENER_API_TOKEN`** from **`week 4/.env`**. Do not commit `.env`.

Calls CourtListener **Legal Search API v4**: `GET /api/rest/v4/search/` with `type=o` (opinion clusters).

The API returns **`caseName`**, **`judge`**, and **`dateFiled`**. Plaintiff and defendant are parsed from `caseName` when it looks like *Party A v. Party B*; otherwise those cells are missing (`NA`).

Install: `python3 -m pip install -r "week 4/requirements.txt"` (same interpreter as this notebook).

A second table (**Judgments classified by Judge names and jurisdiction**) uses **20** rows from `court_id:wawd` (Western District of Washington / Seattle-area federal) with **`jurisdiction`** = **`Seattle jurisdiction`**.

A third block (**Most recent judgments**) loads King County–related opinions via **`court_id:washctapp`** + **`"King County"`**, sorts by decision date, and shows the **20 newest**.

In [ ]:
from __future__ import annotations

import json
import os
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd
from datetime import date
from IPython.display import Markdown, display

TOKEN_ENV = "COURTLISTENER_API_TOKEN"
SEARCH_URL = "https://www.courtlistener.com/api/rest/v4/search/"


def find_week4_dotenv() -> Path:
    """Resolve `week 4/.env` whether the kernel cwd is repo root or `week 4/`."""
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidate = base / "week 4" / ".env"
        if candidate.is_file():
            return candidate
        if base.name == "week 4" and (base / ".env").is_file():
            return base / ".env"
    return cwd / "week 4" / ".env"


def load_dotenv_file(path: Path) -> None:
    """Load KEY=value from `.env` into os.environ (file wins over existing values)."""
    if not path.is_file():
        return
    text = path.read_text(encoding="utf-8")
    if text.startswith("\ufeff"):
        text = text[1:]
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key = key.strip()
        if key.lower().startswith("export "):
            key = key[7:].strip()
        val = val.strip().strip('"').strip("'")
        if key:
            os.environ[key] = val


def split_parties(case_name: str) -> tuple[str, str]:
    if not case_name or not isinstance(case_name, str):
        return "", ""
    for sep in (" v. ", " V. ", " vs. ", " VS. ", " v ", " V "):
        if sep in case_name:
            left, right = case_name.split(sep, 1)
            return left.strip(), right.strip()
    return "", ""


def courtlistener_search(query: str, *, opinion_type: str = "o", token: str | None) -> dict:
    params = {"q": query, "type": opinion_type}
    url = SEARCH_URL + "?" + urllib.parse.urlencode(params)
    headers = {"Accept": "application/json; indent=2"}
    if token:
        headers["Authorization"] = f"Token {token}"
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", errors="replace")[:800]
        raise RuntimeError(f"CourtListener HTTP {e.code}: {detail}") from e


ENV_PATH = find_week4_dotenv()
load_dotenv_file(ENV_PATH)
token = (os.environ.get(TOKEN_ENV) or "").strip()
if not token:
    raise RuntimeError(
        f"Missing non-empty {TOKEN_ENV}. Set it in {ENV_PATH} (see .env.example). "
        "Do not commit .env."
    )

QUERY = "Miranda v. Arizona"
raw = courtlistener_search(QUERY, token=token)
results = raw.get("results") or []
print("total matches:", raw.get("count"))
print("rows on first page:", len(results))

rows: list[dict[str, object]] = []
for item in results[:25]:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    plaintiff, defendant = split_parties(case)
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    date_dec = item.get("dateFiled") or item.get("dateArgued")

    rows.append(
        {
            "case": case,
            "plaintiff": plaintiff or pd.NA,
            "defendant": defendant or pd.NA,
            "judge": judge or pd.NA,
            "date of decision": date_dec or pd.NA,
        }
    )

df = pd.DataFrame(rows)
display(df)

display(
    Markdown(
        "## Judgments classified by Judge names and jurisdiction\n\n"
        "Federal Seattle-area opinions: `court_id:wawd` (W.D. Wash.). "
        "Column **jurisdiction** is the label **Seattle jurisdiction** for each row."
    )
)

SEATTLE_JURISDICTION_LABEL = "Seattle jurisdiction"
raw_seattle = courtlistener_search("court_id:wawd", token=token)
results_seattle = raw_seattle.get("results") or []
print("Seattle-area (W.D. Wash.) total matches:", raw_seattle.get("count"))
print("rows used (first 20):", min(20, len(results_seattle)))

rows_jurisdiction: list[dict[str, object]] = []
for item in results_seattle[:20]:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    court_name = (item.get("court") or "").strip()
    court_id = (item.get("court_id") or "").strip()
    date_dec = item.get("dateFiled") or item.get("dateArgued")

    rows_jurisdiction.append(
        {
            "judge": judge or pd.NA,
            "jurisdiction": SEATTLE_JURISDICTION_LABEL,
            "court": court_name or pd.NA,
            "court_id": court_id or pd.NA,
            "case": case or pd.NA,
            "date of decision": date_dec or pd.NA,
        }
    )

df_jurisdiction = pd.DataFrame(rows_jurisdiction)
display(df_jurisdiction)

display(
    Markdown(
        "## Most recent judgments (King County)\n\n"
        "CourtListener has no separate `court_id` for King County Superior Court. "
        "We use **Court of Appeals of Washington** (`washctapp`) plus the phrase **\"King County\"**, "
        "paginate search results, sort by **`dateFiled`** descending, and take the **top 20**."
    )
)


def _opinion_date(item: dict) -> date:
    ds = item.get("dateFiled") or ""
    parts = ds.split("-")
    if len(parts) != 3:
        return date.min
    y, m, d = (int(parts[0]), int(parts[1]), int(parts[2]))
    return date(y, m, d)


def collect_search_pages(q: str, *, token: str, max_pages: int = 25) -> list[dict]:
    headers = {"Accept": "application/json; indent=2", "Authorization": f"Token {token}"}
    url = SEARCH_URL + "?" + urllib.parse.urlencode({"q": q, "type": "o"})
    acc: list[dict] = []
    for _ in range(max_pages):
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=60) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
        acc.extend(payload.get("results") or [])
        url = payload.get("next")
        if not url or len(acc) >= 1200:
            break
    return acc


QUERY_KING_COUNTY = '"King County" court_id:washctapp'
print(
    "King County–indexed matches (Wash. Ct. App.):",
    courtlistener_search(QUERY_KING_COUNTY, token=token).get("count"),
)
king_items = collect_search_pages(QUERY_KING_COUNTY, token=token)
king_items.sort(key=_opinion_date, reverse=True)
king_top20 = king_items[:20]
print("rows paginated:", len(king_items), "· newest by dateFiled:", len(king_top20))

rows_recent: list[dict[str, object]] = []
for item in king_top20:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    rows_recent.append(
        {
            "case": case or pd.NA,
            "judge": judge or pd.NA,
            "court": (item.get("court") or "").strip() or pd.NA,
            "court_id": (item.get("court_id") or "").strip() or pd.NA,
            "date filed": item.get("dateFiled") or pd.NA,
        }
    )

df_recent = pd.DataFrame(rows_recent)
df_recent

### Notes

- **Pagination**: follow the `next` URL for more pages.
- **King County**: **Most recent judgments** uses `court_id:washctapp` + `"King County"` because King County Superior Court is not its own CourtListener court id.
- **Docs**: [Legal Search API](https://www.courtlistener.com/help/api/rest/search/)